In [1]:
import pandas as pd
import os
import numpy as np
import seaborn as sns

In [2]:
from matplotlib import pyplot as plt
from shapely import geometry

In [3]:
from scipy.stats import pearsonr as cor

In [4]:
import geopandas as gpd

In [5]:
base = os.path.join(os.getcwd(),'..','compiled values')

In [6]:
shapeFile = os.path.join(os.getcwd(),'..','shape','States_shapefile.shp')

In [7]:
shape = gpd.read_file(shapeFile)

In [8]:
shape.drop(columns=['FID', 'Program', 'State_Code','Flowing_St', 'FID_1'],inplace=True)

In [9]:
shape.rename(columns={'State_Name':'State'},inplace=True)

In [10]:
lp_abgm_ir = pd.read_excel(os.path.join(base,'LAI_irrigated_compiled.xlsx'))
lp_abgm_rf = pd.read_excel(os.path.join(base,'LAI_rainfed_compiled.xlsx'))
rs_gpp = pd.read_excel(os.path.join(base,'rs_lai_compiled.xlsx'))

In [11]:
lp_abgm_ir.columns

Index(['Unnamed: 0', 'State_Name', 'Mean(SO)', 'date'], dtype='object')

In [12]:
gpp_grp = os.path.join(base,'gpp graphs')
os.makedirs(gpp_grp,exist_ok=True)

In [13]:
# rs_gpp.drop(columns=['Unnamed: 0'],inplace=True)
rs_gpp.rename(columns={'mean':'LAI'},inplace=True)

In [14]:
yield_data = pd.read_excel(os.path.join(base,'Corn_apy_data.xlsx'))

In [15]:

lp_abgm_ir.rename(columns={'Mean(SO)':'LAI(IR)','State':'State_Name'},inplace=True)
lp_abgm_ir['year']=lp_abgm_ir['date'].apply(lambda x:x.year)
lp_abgm_ir['month']=lp_abgm_ir['date'].apply(lambda x:x.month)
lp_abgm_ir.drop(columns=['Unnamed: 0','date'],inplace=True)

lp_abgm_rf.rename(columns={'Mean(SO)':'LAI(RF)','State':'State_Name'},inplace=True)
lp_abgm_rf['year']=lp_abgm_rf['date'].apply(lambda x:x.year)
lp_abgm_rf['month']=lp_abgm_rf['date'].apply(lambda x:x.month)
lp_abgm_rf.drop(columns=['Unnamed: 0','date'],inplace=True)

In [16]:
def gen_index(df):
    df['ival']=df['State_Name']+'-'+df['year'].astype(str)+'-'+df['month'].astype(str)
    df.drop(columns=['State_Name','year', 'month'],inplace=True)
    return df

In [17]:
lp_abgm_rf = gen_index(lp_abgm_rf)
lp_abgm_ir = gen_index(lp_abgm_ir)
rs_gpp = gen_index(rs_gpp)

In [18]:
# lp_abgm_rf.drop(columns=['index'],inplace=True)
# lp_abgm_ir.drop(columns=['index'],inplace=True)

In [19]:
lp_abgm_rf.columns

Index(['LAI(RF)', 'ival'], dtype='object')

In [20]:
gpp_compiled = pd.merge(lp_abgm_rf,lp_abgm_ir,on='ival')
gpp_compiled = pd.merge(gpp_compiled,rs_gpp,on='ival')

In [21]:
gpp_compiled['year']=gpp_compiled['ival'].apply(lambda x:int(x.split('-')[1]))
gpp_compiled['month']=gpp_compiled['ival'].apply(lambda x:int(x.split('-')[2]))
gpp_compiled['State']=gpp_compiled['ival'].apply(lambda x:(x.split('-')[0]))
gpp_compiled = gpp_compiled.set_index('ival')

In [22]:
gpp_compiled.to_excel(os.path.join(base,'LAI_compiled.xlsx'))

In [23]:
gpp_compiled['LAI']=gpp_compiled['LAI']*0.1

In [24]:
gpp_cv=  gpp_compiled.groupby(['State','month']).mean()

In [25]:
gpp_cv = gpp_cv.reset_index()

In [26]:
gpp_cv.drop(columns='year',inplace=True)

In [27]:
grp = gpp_cv.groupby('month')
agbm_rf = pd.DataFrame()
agbm_ir =  pd.DataFrame()
gpp =  pd.DataFrame()
c=0
for d in grp:
    fn = os.path.join(gpp_grp,str(d[0])+'_gpp.shp')
    gdf = shape.merge(d[1],on='State')

    rf = d[1][['State','LAI(RF)']]
    rf.rename(columns={'LAI(RF)':d[0]},inplace=True)
    rf.fillna(0,inplace=True)
    
    ir = d[1][['State','LAI(IR)']]
    ir.rename(columns={'LAI(IR)':d[0]},inplace=True)
    ir.fillna(0,inplace=True)
    
    gp = d[1][['State','LAI']]
    gp.rename(columns={'LAI':d[0]},inplace=True)
    gp.fillna(0,inplace=True)

    if c!=0:
        agbm_rf = pd.merge(agbm_rf,rf,on='State')
        agbm_ir = pd.merge(agbm_ir,ir,on='State')
        gpp = pd.merge(gpp,gp,on='State')
    else:
        agbm_rf = rf
        agbm_ir = ir
        gpp = gp
        c+=1
    
    if not os.path.exists(fn):
        gdf.to_file(fn)

In [28]:
gpp_gdf = shape.merge(gpp,on='State',how='left')
agbm_rf_gdf = shape.merge(agbm_rf,on='State',how='left')
agbm_ir_gdf = shape.merge(agbm_ir,on='State',how='left')

In [29]:
cols = range(1,13)

In [30]:
# fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(55, 70))
# axes = axes.flatten()

# for i, col in enumerate(cols):
#     gdf = agbm_rf_gdf
#     gdf = gdf[(gdf['State']!='ALASKA') & (gdf['State']!='HAWAII')]
#     gdf.plot(
#         column=col,
#         ax=axes[i],
#         cmap='viridis',
#         legend=True,
#         edgecolor='black',        
#         legend_kwds={
#             'label': col,
#             'orientation': 'vertical'
#         }
#     )    
    
#     cbar = axes[i].get_figure().axes[-1]
#     cbar.tick_params(labelsize=30)
    
#     axes[i].set_title('Month '+str(col), fontsize=50)
#     axes[i].axis('off')
    
# fig.suptitle(
#     "Spatial Distribution of mean LAI (rf) (2010-19)",
#     fontsize=50,
#     fontweight='bold',
#     y=1.02  
# )    
# plt.tight_layout()
# fn = os.path.join(gpp_grp,'SD_LAI_rf.png')
# plt.savefig(fn, bbox_inches='tight',dpi=300)
# plt.show()

In [31]:
# fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(55, 70))
# axes = axes.flatten()

# for i, col in enumerate(cols):
#     gdf = gpp_gdf
#     gdf = gdf[(gdf['State']!='ALASKA') & (gdf['State']!='HAWAII')]
#     gdf.plot(
#         column=col,
#         ax=axes[i],
#         cmap='viridis',
#         legend=True,
#         edgecolor='black',        
#         legend_kwds={
#             'label': col,
#             'orientation': 'vertical'
#         }
#     )    
    
#     cbar = axes[i].get_figure().axes[-1]
#     cbar.tick_params(labelsize=30)
    
#     axes[i].set_title('Month '+str(col), fontsize=50)
#     axes[i].axis('off')
    
# fig.suptitle(
#     "Spatial Distribution of mean LAI (2010-19)",
#     fontsize=50,
#     fontweight='bold',
#     y=1.02  
# )    
# plt.tight_layout()
# fn = os.path.join(gpp_grp,'SD_LAI_rs.png')
# plt.savefig(fn, bbox_inches='tight',dpi=300)
# plt.show()

In [32]:
# fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(55, 70))
# axes = axes.flatten()

# for i, col in enumerate(cols):
#     gdf = agbm_ir_gdf
#     gdf = gdf[(gdf['State']!='ALASKA') & (gdf['State']!='HAWAII')]
#     gdf.plot(
#         column=col,
#         ax=axes[i],
#         cmap='viridis',
#         legend=True,
#         edgecolor='black',        
#         legend_kwds={
#             'label': col,
#             'orientation': 'vertical'
#         }
#     )    
    
#     cbar = axes[i].get_figure().axes[-1]
#     cbar.tick_params(labelsize=30)
    
#     axes[i].set_title('Month '+str(col), fontsize=50)
#     axes[i].axis('off')
    
# fig.suptitle(
#     "Spatial Distribution of mean LAI (ir) (2010-19)",
#     fontsize=50,
#     fontweight='bold',
#     y=1.02  
# )    
# plt.tight_layout()
# fn = os.path.join(gpp_grp,'SD_LAI_ir.png')
# plt.savefig(fn, bbox_inches='tight',dpi=300)
# plt.show()

In [33]:
gpp_compiled.drop(columns=['Unnamed: 0'],inplace=True)

In [34]:
gpp_compiled_cs = gpp_compiled[(gpp_compiled['month']>3) & (gpp_compiled['month']<11)]

In [35]:
gpp_compiled.head()

,LAI(RF),LAI(IR),LAI,year,month,State
ival,,,,,,
ALABAMA-2010-1,0.000132,0.000000,1.600386,2010,1,ALABAMA
ARIZONA-2010-1,0.000000,0.000000,1.104670,2010,1,ARIZONA
ARKANSAS-2010-1,0.000641,0.000005,0.808733,2010,1,ARKANSAS
CALIFORNIA-2010-1,0.003900,0.001765,1.268239,2010,1,CALIFORNIA
COLORADO-2010-1,0.175637,0.779418,0.543978,2010,1,COLORADO


In [36]:
state = []
ir_lai = []
rf_lai = []
for d in gpp_compiled.groupby('State'):
    state.append(d[0])
    d[1].dropna(inplace=True)
    if d[1].shape[0]>2:
        ir_lai.append(cor(d[1]['LAI(IR)'],d[1]['LAI'])[0])
        rf_lai.append(cor(d[1]['LAI(RF)'],d[1]['LAI'])[0])
    else:
        ir_lai.append(np.nan)
        rf_lai.append(np.nan)
df_corr = pd.DataFrame()
df_corr['State'] = state
df_corr['RS-IR'] = ir_lai
df_corr['RS-RF'] = rf_lai

In [37]:
state = []
ir_lai = []
rf_lai = []
for d in gpp_compiled_cs.groupby('State'):
    state.append(d[0])
    d[1].dropna(inplace=True)
    if d[1].shape[0]>2:
        ir_lai.append(cor(d[1]['LAI(IR)'],d[1]['LAI'])[0])
        rf_lai.append(cor(d[1]['LAI(RF)'],d[1]['LAI'])[0])
    else:
        ir_lai.append(np.nan)
        rf_lai.append(np.nan)

df_corr_cs = pd.DataFrame()
df_corr_cs['State'] = state
df_corr_cs['RS-IR(CS)'] = ir_lai
df_corr_cs['RS-RF(CS)'] = rf_lai

In [38]:
gdf_corr = shape.merge(df_corr,on='State',how='left')
gdf_corr = gdf_corr.merge(df_corr_cs,on='State',how='left')

In [39]:
cols_corr = ['RS-IR', 'RS-RF', 'RS-IR(CS)', 'RS-RF(CS)']
gdf_corr.columns

Index(['State', 'geometry', 'RS-IR', 'RS-RF', 'RS-IR(CS)', 'RS-RF(CS)'], dtype='object')

In [40]:
# fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(55, 30))
# axes = axes.flatten()

# for i, col in enumerate(cols_corr):
#     gdf = gdf_corr
#     gdf = gdf[(gdf['State']!='ALASKA') & (gdf['State']!='HAWAII')]
#     gdf.plot(
#         column=col,
#         ax=axes[i],
#         cmap='viridis',
#         legend=True,
#         edgecolor='black',        
#         legend_kwds={
#             'label': col,
#             'orientation': 'vertical'
#         }
#     )    
    
#     cbar = axes[i].get_figure().axes[-1]
#     cbar.tick_params(labelsize=30)
    
#     axes[i].set_title('Correlarion '+str(col), fontsize=50)
#     axes[i].axis('off')
    
# fig.suptitle(
#     "Spatial correlation",
#     fontsize=50,
#     fontweight='bold',
#     y=1.02  
# )    
# plt.tight_layout()
# fn = os.path.join(gpp_grp,'SD_LAI_correlation.png')
# plt.savefig(fn, bbox_inches='tight',dpi=300)
# plt.show()

In [41]:
gdf_corr

,State,geometry,RS-IR,RS-RF,RS-IR(CS),RS-RF(CS)
0,ALABAMA,"POLYGON ((-85.07007 31.98070, -85.11515 31.907...",0.882684,0.919405,0.870772,0.889407
1,ALASKA,"MULTIPOLYGON (((-161.33379 58.73325, -161.3824...",NaN,NaN,NaN,NaN
2,ARIZONA,"POLYGON ((-114.52063 33.02771, -114.55909 33.0...",0.813011,0.840365,0.759851,0.770549
3,ARKANSAS,"POLYGON ((-94.46169 34.19677, -94.45262 34.508...",0.807663,0.948989,0.718170,0.922777
4,CALIFORNIA,"MULTIPOLYGON (((-121.66522 38.16929, -121.7823...",0.898358,0.894599,0.942359,0.932535
5,COLORADO,"POLYGON ((-102.04446 37.64147, -102.04201 37.3...",0.946603,0.961172,0.961799,0.953296
6,CONNECTICUT,"POLYGON ((-73.53039 41.52275, -73.51715 41.665...",NaN,NaN,NaN,NaN
7,DELAWARE,"POLYGON ((-75.70707 38.55759, -75.71071 38.649...",0.909102,0.963075,0.894870,0.961788
8,DISTRICT OF COLUMBIA,"POLYGON ((-77.00793 38.96667, -76.91090 38.890...",NaN,NaN,NaN,NaN
9,FLORIDA,"MULTIPOLYGON (((-80.78566 28.78519, -80.76242 ...",0.728145,0.842583,0.594845,0.707842
